# 02 — Extract ESD index from EDIT MLRA pages

This notebook takes the MLRA catalog produced by the previous notebook and builds the next database layer:

```text
MLRA catalog
    ↓
MLRA page
    ↓
Ecological site index
```

Target output:

| database_year | mlra_symbol | mlra_name | ecoclassid | site_name | site_url | accessed_date |
|---|---|---|---|---|---|---|

This notebook does **not** parse full ESD pages yet. It only discovers site-level records and checks whether the extracted count matches the catalog-reported ecological-site count.


In [3]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm


## Configuration

Set `DATABASE_YEAR` to the annual snapshot year. The raw HTML cache is versioned by year so later STEP builds can reproduce the exact source state used for an ESD database snapshot.


In [5]:
# Cell 2 — base directory

BASE_DIR = Path(r"C:\NCA_DATA\Ancillary_Data\NRCS ESDs")
DATABASE_YEAR = 2026

YEAR_DIR = BASE_DIR / str(DATABASE_YEAR)
RAW_DIR = YEAR_DIR / "raw_html"
TABLE_DIR = YEAR_DIR / "tables"
LOG_DIR = YEAR_DIR / "logs"

for d in [BASE_DIR, YEAR_DIR, RAW_DIR, TABLE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MLRA_CATALOG_CSV = BASE_DIR / "mlra_catalog.csv"

ESD_INDEX_CSV = TABLE_DIR / f"esd_index_{DATABASE_YEAR}.csv"
FETCH_LOG_CSV = LOG_DIR / f"mlra_fetch_log_{DATABASE_YEAR}.csv"

print("Base:", BASE_DIR)
print("Year directory:", YEAR_DIR)

Base: C:\NCA_DATA\Ancillary_Data\NRCS ESDs
Year directory: C:\NCA_DATA\Ancillary_Data\NRCS ESDs\2026


## Load and validate MLRA catalog

The parser requires these fields from the previous notebook:

- `mlra_symbol`
- `mlra_name`
- `mlra_url`
- `ecological_site_count`


In [6]:
# Cell 3 — load MLRA catalog

mlra_catalog = pd.read_csv(MLRA_CATALOG_CSV)

required = {
    "mlra_symbol",
    "mlra_name",
    "mlra_url",
    "ecological_site_count",
}

missing = required - set(mlra_catalog.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

mlra_catalog.head()

,mlra_symbol,mlra_name,edit_unit_id,mlra_url,ecological_site_count,hidden_until_view_all
0,001X,"Northern Pacific Coast Range, Foothills, and V...",27,https://edit.sc.egov.usda.gov/catalogs/esd/001X,53,False
1,002X,Willamette and Puget Sound Valleys,28,https://edit.sc.egov.usda.gov/catalogs/esd/002X,43,False
2,003X,Olympic and Cascade Mountains,29,https://edit.sc.egov.usda.gov/catalogs/esd/003X,79,False
3,004A,Sitka Spruce Belt,30,https://edit.sc.egov.usda.gov/catalogs/esd/004A,31,False
4,004B,Coastal Redwood Belt,31,https://edit.sc.egov.usda.gov/catalogs/esd/004B,65,False


## Fetching with local cache

Each MLRA page is saved once under `raw_html/mlra_pages/`. Re-running the notebook will use the cached HTML unless `force=True` is passed.


In [8]:
# Cell 4 — helpers

HEADERS = {
    "User-Agent": (
        "ESD-index-builder/0.1 "
        "(research workflow)"
    )
}

def clean_text(x):
    if x is None:
        return None
    return re.sub(r"\s+", " ", str(x)).strip()


def html_hash(html):
    return hashlib.sha256(html.encode("utf-8", errors="ignore")).hexdigest()


def fetch_html(url, timeout=60, sleep_seconds=0.75):
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    time.sleep(sleep_seconds)
    return r.text


def raw_mlra_path(mlra_symbol):
    return RAW_DIR / f"mlra_{mlra_symbol}.html"

## Parse ecological-site records from one MLRA page

The parser is intentionally conservative. It looks for links matching:

```text
/catalogs/esd/{MLRA}/{ECOCLASSID}
```

Then it extracts the nearest visible title/name from the surrounding list item. This should be more stable than relying on one specific CSS class.


In [12]:
# Cell 5 — parser for one MLRA page

def parse_mlra_page(html, mlra_symbol, mlra_name, mlra_url):
    soup = BeautifulSoup(html, "html.parser")

    records = []

    # EDIT MLRA pages usually list ecological sites as links containing ecoclass IDs.
    # Ecoclass IDs generally look like R011XY001ID, F011XY..., etc.
    pattern = re.compile(r"/catalogs/esd/[^/]+/([A-Z]\d{3}[A-Z]{2}\d{3}[A-Z]{2})")

    seen = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        m = pattern.search(href)
        if not m:
            continue

        ecoclassid = m.group(1)

        if href.startswith("http"):
            site_url = href
        else:
            site_url = "https://edit.sc.egov.usda.gov" + href

        site_name = clean_text(a.get_text(" ", strip=True))

        # If the link text is only the ID, look nearby for a fuller name.
        parent_text = clean_text(a.find_parent().get_text(" ", strip=True)) if a.find_parent() else site_name

        key = (mlra_symbol, ecoclassid)
        if key in seen:
            continue
        seen.add(key)

        records.append({
            "database_year": DATABASE_YEAR,
            "mlra_symbol": mlra_symbol,
            "mlra_name": mlra_name,
            "ecoclassid": ecoclassid,
            "site_name_from_link": site_name,
            "site_context_text": parent_text,
            "site_url": site_url,
            "mlra_url": mlra_url,
            "accessed_date": date.today().isoformat(),
        })

    return records

## Smoke test: one MLRA

Use a known MLRA first. `011X` is useful because we already inspected `R011XY001ID` manually.


In [15]:
# Cell 6 — fetch/cache MLRA pages and build ESD index

all_records = []
fetch_logs = []

for i, row in mlra_catalog.iterrows():
    mlra_symbol = row["mlra_symbol"]
    mlra_name = row["mlra_name"]
    mlra_url = row["mlra_url"]
    expected_count = int(row["ecological_site_count"])

    out_html = raw_mlra_path(mlra_symbol)

    try:
        if out_html.exists():
            html = out_html.read_text(encoding="utf-8", errors="ignore")
            source = "cache"
        else:
            html = fetch_html(mlra_url)
            out_html.write_text(html, encoding="utf-8")
            source = "web"

        records = parse_mlra_page(html, mlra_symbol, mlra_name, mlra_url)
        all_records.extend(records)

        fetch_logs.append({
            "mlra_symbol": mlra_symbol,
            "mlra_name": mlra_name,
            "mlra_url": mlra_url,
            "expected_site_count": expected_count,
            "parsed_site_count": len(records),
            "count_match": expected_count == len(records),
            "source": source,
            "html_file": str(out_html),
            "html_sha256": html_hash(html),
            "status": "ok",
            "error": None,
        })

    except Exception as e:
        fetch_logs.append({
            "mlra_symbol": mlra_symbol,
            "mlra_name": mlra_name,
            "mlra_url": mlra_url,
            "expected_site_count": expected_count,
            "parsed_site_count": None,
            "count_match": False,
            "source": None,
            "html_file": str(out_html),
            "html_sha256": None,
            "status": "error",
            "error": repr(e),
        })

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1}/{len(mlra_catalog)} MLRAs")

esd_index = pd.DataFrame(all_records)
fetch_log = pd.DataFrame(fetch_logs)

esd_index.to_csv(ESD_INDEX_CSV, index=False)
fetch_log.to_csv(FETCH_LOG_CSV, index=False)

print("Extracted ESD records:", len(esd_index))
print("Expected from MLRA catalog:", int(mlra_catalog["ecological_site_count"].sum()))
print("Count-matching MLRAs:", fetch_log["count_match"].sum(), "/", len(fetch_log))

Processed 25/267 MLRAs
Processed 50/267 MLRAs
Processed 75/267 MLRAs
Processed 100/267 MLRAs
Processed 125/267 MLRAs
Processed 150/267 MLRAs
Processed 175/267 MLRAs
Processed 200/267 MLRAs
Processed 225/267 MLRAs
Processed 250/267 MLRAs
Extracted ESD records: 0
Expected from MLRA catalog: 8300
Count-matching MLRAs: 14 / 267


If `Extracted site count` does not match `Expected site count`, inspect the cached HTML and adjust the parser before running the full catalog.


In [17]:
# Debug one cached MLRA page

test_mlra = "011X"
test_path = raw_mlra_path(test_mlra)

html = test_path.read_text(encoding="utf-8", errors="ignore")
soup = BeautifulSoup(html, "html.parser")

print("HTML chars:", len(html))
print("Title:", soup.title.get_text(" ", strip=True) if soup.title else None)

links = []
for a in soup.find_all("a", href=True):
    href = a["href"]
    text = clean_text(a.get_text(" ", strip=True))
    if "R011" in href or "R011" in text or "/catalogs/esd/011X/" in href:
        links.append((text, href))

len(links), links[:20]

HTML chars: 475581
Title: MLRA 011X


(248,
 [('', 'https://edit.sc.egov.usda.gov/catalogs/esd/011X/R011XA003ID'),
  ('R011XA003ID/R011XA003ID Shallow Loam 8-12 PZ ARTRT/PSSPS',
   'https://edit.sc.egov.usda.gov/catalogs/esd/011X/R011XA003ID'),
  ('Download pdf',
   'https://edit.sc.egov.usda.gov/services/descriptions/esd/011X/R011XA003ID.pdf'),
  ('', 'https://edit.sc.egov.usda.gov/catalogs/esd/011X/R011XA005ID'),
  ('R011XA005ID/R011XA005ID Claypan 8-12 PZ ARTRW8/PSSPS',
   'https://edit.sc.egov.usda.gov/catalogs/esd/011X/R011XA005ID'),
  ('Download pdf',
   'https://edit.sc.egov.usda.gov/services/descriptions/esd/011X/R011XA005ID.pdf'),
  ('', 'https://edit.sc.egov.usda.gov/catalogs/esd/011X/R011XA006ID'),
  ('R011XA006ID/R011XA006ID Saline Upland 7-12 PZ SAVE4/LECI4',
   'https://edit.sc.egov.usda.gov/catalogs/esd/011X/R011XA006ID'),
  ('Download pdf',
   'https://edit.sc.egov.usda.gov/services/descriptions/esd/011X/R011XA006ID.pdf'),
  ('', 'https://edit.sc.egov.usda.gov/catalogs/esd/011X/R011XA007ID'),
  ('R011XA007I

In [16]:
# Cell 7 — inspect mismatches

mismatches = fetch_log.loc[~fetch_log["count_match"]].copy()
mismatches.sort_values(["status", "mlra_symbol"]).head(30)

esd_index.head(20)

""


## Full extraction: MLRA → ESD index

This loops over every MLRA in the catalog. It writes intermediate files so an interrupted run can be resumed.


In [ ]:
all_site_frames = []
fetch_records = []
errors = []

for row in tqdm(mlra_catalog.itertuples(index=False), total=len(mlra_catalog)):
    try:
        fetched = fetch_mlra_html(row.mlra_symbol, row.mlra_url)
        sites = parse_mlra_sites(fetched['html'], row.mlra_symbol, row.mlra_name)
        sites['expected_site_count'] = int(row.ecological_site_count)
        sites['mlra_html_hash'] = fetched['html_hash']
        sites['mlra_html_path'] = fetched['html_path']
        all_site_frames.append(sites)

        fetch_records.append({
            'mlra_symbol': row.mlra_symbol,
            'mlra_name': row.mlra_name,
            'mlra_url': row.mlra_url,
            'expected_site_count': int(row.ecological_site_count),
            'extracted_site_count': len(sites),
            'from_cache': fetched['from_cache'],
            'status_code': fetched['status_code'],
            'html_hash': fetched['html_hash'],
            'html_path': fetched['html_path'],
        })
    except Exception as e:
        errors.append({
            'mlra_symbol': row.mlra_symbol,
            'mlra_name': row.mlra_name,
            'mlra_url': row.mlra_url,
            'error': repr(e),
        })

esd_index = pd.concat(all_site_frames, ignore_index=True) if all_site_frames else pd.DataFrame()
fetch_log = pd.DataFrame(fetch_records)
error_log = pd.DataFrame(errors)

print('Extracted ESD records:', len(esd_index))
print('Catalog expected records:', int(mlra_catalog['ecological_site_count'].sum()))
print('MLRA errors:', len(error_log))


## Completeness checks

The main validation criterion is whether extracted site counts match the catalog counts per MLRA and in total.


In [ ]:
count_check = (
    mlra_catalog[['mlra_symbol', 'mlra_name', 'ecological_site_count']]
    .rename(columns={'ecological_site_count': 'expected_site_count'})
    .merge(
        esd_index.groupby('mlra_symbol').size().rename('extracted_site_count').reset_index(),
        on='mlra_symbol',
        how='left'
    )
)
count_check['extracted_site_count'] = count_check['extracted_site_count'].fillna(0).astype(int)
count_check['count_difference'] = count_check['extracted_site_count'] - count_check['expected_site_count']
count_check['matches_catalog'] = count_check['count_difference'].eq(0)

print('Matching MLRAs:', int(count_check['matches_catalog'].sum()), '/', len(count_check))
print('Total expected:', int(count_check['expected_site_count'].sum()))
print('Total extracted:', int(count_check['extracted_site_count'].sum()))

count_check.loc[~count_check['matches_catalog']].head(20)


In [ ]:
# Duplicate diagnostics
if not esd_index.empty:
    dupes = esd_index[esd_index.duplicated(['mlra_symbol', 'ecoclassid'], keep=False)].sort_values(['mlra_symbol', 'ecoclassid'])
    print('Duplicate MLRA/ecoclassid rows:', len(dupes))
    display(dupes.head(20))


## Save outputs

Outputs are written as CSV and Parquet when possible. CSV is the exchange format; Parquet is faster and preserves types better.


In [ ]:
esd_index_csv = TABLE_DIR / 'esd_index.csv'
count_check_csv = TABLE_DIR / 'esd_index_count_check.csv'
fetch_log_csv = META_DIR / 'mlra_fetch_log.csv'
error_log_csv = META_DIR / 'mlra_error_log.csv'

esd_index.to_csv(esd_index_csv, index=False)
count_check.to_csv(count_check_csv, index=False)
fetch_log.to_csv(fetch_log_csv, index=False)
error_log.to_csv(error_log_csv, index=False)

try:
    esd_index.to_parquet(TABLE_DIR / 'esd_index.parquet', index=False)
    count_check.to_parquet(TABLE_DIR / 'esd_index_count_check.parquet', index=False)
except Exception as e:
    print('Parquet export skipped:', repr(e))

run_metadata = {
    'database_year': DATABASE_YEAR,
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'mlra_count': int(len(mlra_catalog)),
    'catalog_expected_site_count': int(mlra_catalog['ecological_site_count'].sum()),
    'extracted_site_count': int(len(esd_index)),
    'matching_mlra_count': int(count_check['matches_catalog'].sum()),
    'error_count': int(len(error_log)),
    'request_delay_seconds': REQUEST_DELAY_SECONDS,
    'timeout_seconds': TIMEOUT_SECONDS,
    'parser': '02_extract_esd_index.ipynb',
}
(META_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')

print('Saved:')
print(' ', esd_index_csv)
print(' ', count_check_csv)
print(' ', fetch_log_csv)
print(' ', error_log_csv)
print(' ', META_DIR / 'run_metadata.json')


## Next notebook

After this notebook passes the completeness check, the next notebook should parse full ESD pages:

```text
03_extract_esd_detail_pages.ipynb
```

Input:

```text
esd_index.csv
```

First output target:

```text
raw_html/esd_pages/{ecoclassid}.html
parsed_json/{ecoclassid}.json
```

Do not normalize state-transition edges until raw section/table extraction is stable across multiple MLRAs.
